[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlatcl/multilogue/blob/main/notebooks/06_validate.ipynb)

# Validate and report — agreement, sheets, evidence

### [Neil D. Lawrence](http://inverseprobability.com), University of Cambridge

### 2026-10-18

**Abstract**: Validation scaffolding is deliberately thin in the library: stratified sampling
for blind hand-check sheets, Cohen's $\kappa$ and rank agreement helpers, and an
HTML evidence-pack shell. Coding schemes and playbooks stay with the study. This
notebook teaches those tools on synthetic labels and phrases.


In [ ]:
#@title Setup (run first)
# Infrastructure only: install or locate multilogue, prepare figure directory.
# Scientific imports live in later cells.

import sys, subprocess, pathlib

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "matplotlib", "pandas", "numpy", "scikit-learn", "pyyaml", "pot",
        "git+https://github.com/mlatcl/multilogue.git",
    ])
else:
    try:
        import multilogue  # noqa: F401
    except ImportError:
        _here = pathlib.Path.cwd().resolve()
        for _cand in (_here, _here.parent, *_here.parents):
            if (_cand / "pyproject.toml").is_file() and (_cand / "multilogue").is_dir():
                if str(_cand) not in sys.path:
                    sys.path.insert(0, str(_cand))
                break

from multilogue.notebook import setup

FIG = setup(figures_dir="outputs/notebook_figures")
print("Figures →", FIG)


# Introduction

When two codings of the same items exist (human–model or model–model), Cohen's
$\kappa$ summarises chance-corrected agreement. It is a diagnostic, not a
substitute for reading the sheet. Stratified samples give researchers a
manageable blind-check set; the evidence pack is a readable HTML shell over
phrase tables.


In [ ]:
from IPython.display import display, HTML
import numpy as np
import pandas as pd
from multilogue.access import load
from multilogue.validate import stratified_sample
from multilogue.weight import kappa, rank_agreement
from multilogue.report import evidence_pack, write_manifest
from multilogue.dummy import FakeClient
from multilogue.extract import make_extract_work, phrase_table
from multilogue.llm import StageRunner
from multilogue.cluster import (
    Embedder, frozen_or_new_assignments, example_phrases_for_clusters,
    make_label_work, make_lens_work, resolve_lenses,
)


In [ ]:
a = ["A", "A", "B", "B", "A", "B", "A", "B"]
b = ["A", "B", "B", "B", "A", "B", "A", "A"]
print("kappa:", round(kappa(a, b), 3))

main = pd.Series([10.0, 8.0, 6.0, 4.0], index=list("WXYZ"))
other = pd.Series([9.5, 7.0, 6.5, 3.0], index=list("WXYZ"))
print("rank_agreement:", rank_agreement(main, other))


In [ ]:
units = load("synthetic", n_docs_per_group=2, units_per_doc=3)["units"]
units_by_id = {u["unit_id"]: u for u in units}
client = FakeClient()
root = FIG.parent
runner = StageRunner(root / "ckpt_v", root / "logs_v", max_workers=2, sleep=lambda _: None)
extraction = {}
for kind in ("concern", "benefit"):
    work = make_extract_work(
        client, kind=kind, system="e",
        user_for_unit=lambda u, k=kind: u["text"], model="fake-small",
        sentinel=f"NO_{kind.upper()}",
    )
    extraction[kind] = runner.run(
        f"ex_{kind}", units, lambda u: u["unit_id"], work, f"ex|{kind}", quiet=True,
    )
phrases = phrase_table(
    extraction, units_by_id, kinds=["concern", "benefit"], extra_unit_fields=["group"],
)
k = 3
E = np.asarray(Embedder(root / "emb", client, "fake-emb")(phrases.phrase.tolist()), float)
labels, centroids, _ = frozen_or_new_assignments(phrases, E, k, 0, root / "cl")
phrases = phrases.copy()
phrases["cluster"] = labels
phrases["centroid_sim"] = (E * centroids[labels]).sum(1)
cluster_units = example_phrases_for_clusters(phrases, k, 0)
lab = runner.run(
    "lab", cluster_units, lambda cu: cu["key"],
    make_label_work(
        client, system="l", prompt_template="Label:\n<<PHRASES>>", model="fake-small",
    ),
    "lab", quiet=True,
)
names = {
    cu["cluster"]: lab.get(cu["key"], {}).get("label", f"c{cu['cluster']}")
    for cu in cluster_units
}
cl_desc = "\n".join(f"{c}: {names[c]}" for c in range(k))
lens_runs = runner.run(
    "lens", [1], lambda r: str(r),
    make_lens_work(
        client, system="L", prompt_template="Group:\n<<CLUSTERS>>",
        cluster_description=cl_desc, model="fake-small", k=k,
    ),
    "lens", quiet=True,
)
phrases["lens"] = phrases["cluster"].map(resolve_lenses(k, lens_runs)[0])

sheet = stratified_sample(phrases, by="group", n_per_stratum=2, random_state=0)
display(sheet[["unit_id", "group", "phrase", "lens"]])
html = evidence_pack(
    phrases, root / "evidence_pack.html", group_col="group",
    title="Synthetic evidence pack", seed=0,
)
write_manifest(root / "manifest.json", {"n_phrases": len(phrases), "synthetic": True})
print("Evidence pack:", html)
display(HTML(f'<a href="{html}" target="_blank">Open evidence pack</a>'))


The stratified sheet is what a researcher would code blind. The evidence pack is
a navigable HTML view of the same phrase table — useful for reading, not a
substitute for $\kappa$ or for substantive judgment.


# Outlook

Notebook 07 assembles notebooks 00–06 into one offline smoke experiment.
Notebook 08 develops the soft geometry under Multilogue's lens model.
